## Configuration

In [ ]:
from pathlib import Path
import tomllib
from utils.utils import *
import json

PATH_EEG = Path("../EEG/EEG/results/")
FILENAME = 'Case_09_ExG.csv'
MARKER_FILE = 'Case_09_Marker.csv'
CONFIG_PATH = "../EEG/EEG/config/config.toml"

with open(CONFIG_PATH, "rb") as f:
    config = tomllib.load(f)

BANDS = config['BANDS']
SEGMENT_DEF = config['SEGMENT_DEF']
RANDOM_SEED = config['RANDOM_SEED']
DURATION = config['DURATION']

SFREQ = config['SFREQ']
L_FREQ = config['L_FREQ']
H_FREQ = config['H_FREQ']

## Raw Data

In [ ]:
raw_full, df_full, df_markers_full = load_and_preprocess_eeg(PATH_EEG, FILENAME, MARKER_FILE, SFREQ, L_FREQ, H_FREQ)

raws, labels = extract_segments(raw_full, df_full, df_markers_full, SEGMENT_DEF)

In [ ]:
plot_psd_comparison(raws, labels)

In [ ]:
plot_topomap_comparison(raws, labels, band_name='Alpha')
plot_topomap_comparison(raws, labels, band_name='Beta')

In [ ]:
plot_topomap_comparison(raws, labels, band_name='Delta')
plot_topomap_comparison(raws, labels, band_name='Theta')

## Preprocessing

In [ ]:
import mne
from mne.preprocessing import ICA
raw, df, df_markers = load_and_preprocess_eeg(PATH_EEG, FILENAME, MARKER_FILE, SFREQ, L_FREQ, H_FREQ)

# ICA
# n_components=15 (16 Channels, ICA finding max n_channels - 1 or same number)
ica = ICA(n_components=15, max_iter='auto', random_state=RANDOM_SEED)

print(">>> ICA (Might take a while)...")
ica.fit(raw)

print(">>> Showing Components...")
ica.plot_components()
plt.show()

ica.plot_sources(raw, show_scrollbars=True)
plt.show()

In [ ]:
exclude_indices = [3, 4, 9, 14]

print(f">>> Removing ingredients: {exclude_indices}")
ica.exclude = exclude_indices

raw_clean = raw.copy()
ica.apply(raw_clean)

print(">>> Cleaned data.")

target_chan = "F3" if "F3" in raw.ch_names else raw.ch_names[0]

print(f"Comparison on the channel {target_chan}...")
fig, ax = plt.subplots(figsize=(15, 6))
ax.plot(
    raw.get_data(picks=target_chan)[0, 1000:2500],
    color="red",
    alpha=0.5,
    label="Before cleaning (Original)",
)
ax.plot(
    raw_clean.get_data(picks=target_chan)[0, 1000:2500],
    color="black",
    label="After cleaning (Clean)",
)
ax.set_title(f"Artifact Removal Effect (Eyes and Muscles) - Channel {target_chan}")
ax.legend()
plt.show()

In [ ]:
raws, labels = extract_segments(raw_clean, df_full, df_markers_full, SEGMENT_DEF)

plot_psd_comparison(raws, labels)

In [ ]:
plot_topomap_comparison(raws, labels, band_name='Alpha')
plot_topomap_comparison(raws, labels, band_name='Beta')

In [ ]:
plot_topomap_comparison(raws, labels, band_name='Delta')
plot_topomap_comparison(raws, labels, band_name='Gamma')

#### Analiza Pasma Alpha (8–12 Hz):
Na topomapach brak wyraźnej dysocjacji C3 vs C4. Cały pas centralny (motoryczny) pozostaje całkowicie wyciszony (granatowy) w każdym z trybów. Oznacza to, że sam ruch palcem nie generuje widocznej na mapie zmiany mocy względem tła.

Fazy 1, 2 i 4: Mózg jest "pusty" (granatowy). Pasywne oglądanie i tryb Smart nie generują niemal żadnej widocznej mocy fal Alpha.

Faza 3 (Brainrot Scroll): Pojawia się potężna, zlokalizowana Alpha z tyłu głowy (O1/O2, czerwona plama). Uczestnik wchodzi w stan silnego pobudzenia wizualnego połączonego z zawieszeniem uwagi w momencie przewijania szybkich treści.

#### Analiza Pasma Beta (13–30 Hz):
Cały mózg pozostaje wyciszony (Niebieski), z wyjątkiem kory potylicznej (O1 i O2), która jest ekstremalnie aktywna (Czerwona) wyłącznie w fazie Brainrot Scroll.

Uczestnik nie filtruje bodźców podczas treści Smart (tam mózg "śpi"). Najsilniejsza aktywacja występuje przy mechanicznym scrollowaniu Brainrotu – kora wzrokowa pracuje na najwyższych obrotach, próbując przetworzyć migające klatki nowych, szybko zmieniających się filmików. Tryb Smart nie wymaga u niego żadnej pracy analitycznej Bety.

#### Analiza Pasma Delta (1–4 Hz):
Hierarchia Wybudzenia: 
* Faza 1, 2 i 4: Delta jest bliska zera (granat). Brak oznak "transu" czy obciążenia w tych trybach.
* Faza 3 (Brainrot Scroll): Anomalia w paśmie Delta w potylicy (lewy dolny róg). Zwykle silna Delta z tyłu głowy u osoby przytomnej to potężne przebodźcowanie, skrajne zmęczenie analizą wizualną lub wynik silnych artefaktów.

To nie "Smart" wymusza przytomność. To wyłącznie "Brainrot Scroll" generuje jakiekolwiek znaczące fale w mózgu tego uczestnika.

#### Analiza Pasma Gamma (>30 Hz):
Scrollowanie: Gwałtowny wzrost Gammy występuje tylko i wyłącznie w fazie Brainrot Scroll (lewy dolny róg). Smart Scroll pozostaje całkowicie ciemny.
Koszt fizjologiczny ponosi kora wzrokowa (O1/O2) z powodu ekstremalnie szybkiego tempa zmian obrazu w trybie Brainrot.

Dominacja Fazy 3: Mechaniczne scrollowanie brainrotu dosłownie "przepala" korę wzrokową badanego, zmuszając ją do pracy integrującej bodźce. Czytanie ze zrozumieniem (Smart) nie wywołuje u niego absolutnie żadnego obciążenia.

---

Znieczulenie na "Smart": Uczestnik kompletnie nie angażuje się w trudniejsze/wartościowe treści. Niezależnie czy patrzy, czy scrolluje tryb Smart, jego mózg pozostaje "wyłączony" na wykresach topograficznych (brak widocznej mocy w badanych pasmach).

Jedyne momenty, w których badany wykazuje gigantyczną aktywność układu nerwowego (rozpalona potylica we wszystkich pasmach), to chwile mechanicznego przewijania szybkich, prostych treści (Brainrot Scroll).

Gdy brakuje natychmiastowych bodźców typu "Brainrot", jego kora mózgowa obniża aktywność do absolutnego minimum.

## Spektogram

In [ ]:
# Alpha (fatigue/relax)
plot_band_power_over_time(raws, labels, band=(8, 12), band_name='Alpha')
# Beta (Focus)
plot_band_power_over_time(raws, labels, band=(13, 30), band_name='Beta')

In [ ]:
# Emotion
calculate_asymmetry(raws, labels, ch_left='F3', ch_right='F4')

# According to Heller's model, parietal asymmetry is responsible for physiological arousal
calculate_asymmetry(raws, labels, ch_left='P3', ch_right='P4')

# Image processing method.
calculate_asymmetry(raws, labels, ch_left='O1', ch_right='O2')

# Motor
calculate_asymmetry(raws, labels, ch_left='C3', ch_right='C4')

### Podsumowanie

#### Kora Czołowa (F4 vs F3) – Głębokie wycofanie i awersja
Wszystkie słupki mają wartości ujemne.

Ten uczestnik wykazuje bardzo silną reakcję wycofania. Zamiast motywacji dążeniowej i nagrody, jego płaty czołowe mówią: "nie podoba mi się to, jestem znudzony/sfrustrowany". Co ciekawe, najgłębszą awersję wywołuje u niego pasywne oglądanie "Brainrotu" oraz samo przewijanie mądrzejszych treści ("Smart Scroll").

#### Kora Ruchowa (C4 vs C3)
U poprzednich badanych słupki były dodatnie (kciuk prawej ręki aktywował lewą półkulę). U tego badanego pojawiają się silne wartości ujemne.

Ujemny wskaźnik oznacza, że kora C4 (prawa półkula) jest znacznie bardziej aktywna niż C3. A prawa półkula steruje lewą stroną ciała. Istnieje ogromne prawdopodobieństwo, że ten badany jest osobą leworęczną (trzymał telefon i scrollował lewą ręką)! Jeśli nie jest leworęczny, oznacza to, że jego prawa ręka była całkowicie zablokowana/napięta z nerwów, a lewa strona ciała przejęła ładunek stresu.

#### Kora Ciemieniowa (P4 vs P3) – Napięcie przestrzenne i stres
Dominacja prawej kory ciemieniowej jest silnie powiązana z przetwarzaniem przestrzennym, ale też z wyższym poziomem negatywnego pobudzenia (arousal) i przebodźcowaniem.

Aplikacja stresuje układ nerwowy. Co ciekawe, w trybie "Smart" (pasywne oglądanie) ten stres niemal spada do zera (słupek jest prawie niewidoczny). Niestety, tryb "Brainrot" (zarówno oglądanie, jak i scrollowanie) wyraźnie obciąża go przestrzennie.

#### Kora Potyliczna (O2 vs O1)
Pasywny Brainrot: Ogromna dominacja lewej półkuli (O1). Gdy badany tylko patrzy na szybkie filmiki, włącza tryb "mikroskopu". Prawdopodobnie z wielkim wysiłkiem próbuje analizować detale w tym filmiku.

Brainrot Scroll: W momencie, gdy wykonywany jest ruch palcem, następuje drastyczne tąpnięcie. Przewijanie ekranu w tym trybie całkowicie niszczy zdolność do skupienia się na detalach. Zamazany obraz w trakcie szybkiego scrollowania powoduje "szok wizualny" – zmusza prawą półkulę (odpowiedzialną za orientację przestrzenną) do awaryjnego zorientowania się, co się w ogóle dzieje na ekranie.

W trybie Smart wzrok pozostaje stabilny i lekko analityczny (dodatni) niezależnie od tego, czy przewija, czy patrzy.

## Scroll

In [ ]:
PROJECT_ROOT = Path.cwd().parent 

USER_MAP_PATH = PROJECT_ROOT / "EEG" / "EEG" / "user_map.json"
INTERACTIONS_PATH = PROJECT_ROOT / "EEG" / "research_events_rows.csv"
USERS_PATH = PROJECT_ROOT / "EEG" / "Users_rows.csv"

FILE_NAME = 'Case_09_ExG.csv'

In [ ]:
scroll_data = extract_scroll_events(FILE_NAME, INTERACTIONS_PATH, USERS_PATH, USER_MAP_PATH)

if scroll_data is not None and not scroll_data.empty:
    new_annotations = mne.Annotations(
        onset=scroll_data['onset'].values,
        duration=scroll_data['duration'].values,
        description=scroll_data['description'].values
    )

    raw_clean.set_annotations(new_annotations)
    
    print("SUCCESS: Annotations applied to raw_clean.")
    print(f"Time range of scrolls: {scroll_data['onset'].min():.2f}s to {scroll_data['onset'].max():.2f}s")
else:
    print("WARNING: No scroll data found. Plots will be empty.")

In [ ]:
compare_stages(
    raw_clean, 
    band='Beta', 
    fmin=13, 
    fmax=30, 
    channel='C3',  
    duration=DURATION
)

In [ ]:
# REWARD/MOTIVATION SYSTEM (Frontal Alpha)
# Channel: F3 (Left Frontal)
# Band: Alpha (8-12 Hz)
# Interpretation: THE LOWER THE GRAPH, THE GREATER THE “REWARD/ENGAGEMENT.”

compare_stages(
    raw_clean, 
    band='Alpha', 
    fmin=8, 
    fmax=12, 
    channel='F3',   # Key channel for positive/aspirational emotions
    duration=DURATION
)

In [ ]:
# ISUAL RELAXATION (Alpha)
# Channel: O1 (Back of the head)
# Band: Alpha (8-12 Hz)
# Interpretation: HIGH LINE = RELAXATION / ZOMBIE MODE. LOW = WORK.

compare_stages(
    raw_clean, 
    band='Alpha', 
    fmin=8, 
    fmax=12, 
    channel='O1', 
    duration=DURATION
)

In [ ]:
# INTELLECTUAL EFFORT (Frontal Theta) ---
# Channel: Fz (Center of forehead)
# Band: Theta (4-8 Hz)
# Interpretation: PEAKS = STRONG THINKING/MEMORIZATION.

target_channel = 'Fz'

compare_stages(
    raw_clean, 
    band='Theta', 
    fmin=4, 
    fmax=8, 
    channel=target_channel, 
    duration=100
)

In [ ]:
# COGNITIVE CONTROL & INHIBITION (Right Frontal Theta) ---
# Channel: F4 (Right forehead / Right Frontal)
# Band: Theta (4-8 Hz)
# Interpretation: PEAKS = SUSTAINED FOCUS & SUPPRESSING DISTRACTIONS (Mental effort to stay on task or regulate emotional impulses).

target_channel = 'F4'

compare_stages(
    raw_clean, 
    band='Theta', 
    fmin=4, 
    fmax=8, 
    channel=target_channel, 
    duration=DURATION
)

In [ ]:
# VERBAL & ANALYTICAL MEMORY (Left Parietal Theta) ---
# Channel: P3 (Left back of head / Left Parietal)
# Band: Theta (4-8 Hz)
# Interpretation: PEAKS = PROCESSING TEXT/LANGUAGE (Reading captions, understanding speech, verbal working memory).

target_channel = 'P3'

compare_stages(
    raw_clean, 
    band='Theta', 
    fmin=4, 
    fmax=8, 
    channel=target_channel, 
    duration=DURATION
)

In [ ]:
# VISUOSPATIAL PROCESSING & ATTENTION (Right Parietal Theta) ---
# Channel: P4 (Right back of head / Right Parietal)
# Band: Theta (4-8 Hz)
# Interpretation: PEAKS = ENCODING VISUAL/SPATIAL INFO (Processing layout, movement, spatial memory, or complex imagery).

target_channel = 'P4'

compare_stages(
    raw_clean, 
    band='Theta', 
    fmin=4, 
    fmax=8, 
    channel=target_channel, 
    duration=DURATION
)

#### Kora wzrokowa (Alpha O1)
**Brainrot Scroll**: Skala wykresu, gdzie piki sięgają tu 800 µV, jest potężną anomalią (fale Alpha mają 10-40 µV). Oznacza to, że badany w trakcie oglądania prawdopodobnie silnie mrużył oczy, zaciskał szczękę z irytacji lub napinał kark.

**Wpływ scrolla**: Sygnał Alpha natychmiastowo "szoruje po dnie". Mechanizm pędzącego obrazu w Brainrocie powoduje drastyczny szok wizualny i całkowite, awaryjne zablokowanie kory potylicznej.

**Smart Scroll**:  Po niebieskich liniach scrollowania fala Alpha po prostu spada i utrzymuje się na niższym poziomie. Oznacza to, że przy statycznym tekście/obrazku wzrok potrafi się względnie ustabilizować i podjąć próbę czytania.

2. Płaty czołowe (Theta Fz, F4) – Frustracja zamiast zrozumienia
**Brainrot Scroll**: Gęste czerwone linie na wykresach Fz i F4. Wykresy są poszarpane i chaotyczne. Nie ma żadnej logicznej reakcji na ruch palcem. Płaty czołowe są zalewane szumem, a uwaga nie potrafi się zakotwiczyć na żadnym konkretnym filmiku.

**Smart Scroll**: Nawet przy rzadszym, wolniejszym przewijaniu (niebieskie linie), po scrollu nie widać wyraźnego momentu skupienia (górki). Mózg jest tak znużony (ujemna asymetria), że nowe treści przyjmuje z obojętnością i ciągłym, lekkim "rozproszeniem".

#### Płaty ciemieniowe (Theta P3, P4)
Kora ciemieniowa (szczególnie prawa - P4) odpowiada za orientację przestrzenną i integrację bodźców.

Na obu wykresach (Brainrot i Smart dla P4) fale Theta są nienaturalnie wysokie (skaczą do 20-30 µV). Linie przewijania wpadają w środek potężnych, ciągłych wahań.

Samo fizyczne trzymanie telefonu, śledzenie znikającego/pojawiającego się obrazu i przewijanie jest dla układu przestrzennego ekstremalnie wyczerpujące. Mózg nie ma tu ani ułamka sekundy na reset orientacji.

#### Kora ruchowa (Beta C3)
Na wykresach C3 (niezależnie czy to Smart, czy Brainrot) linia fal Beta jest poszarpana.

Nie ma tu zjawiska ERD/ERS (czyli wyraźnego spadku przed scrollem i wyskoku zaraz po nim). Oznacza to, że jego prawa kora ruchowa nie wykonuje płynnych, celowych uderzeń kciukiem. Ręka jest w stanie ciągłego, izometrycznego napięcia, drżenia lub dyskomfortu.

---
U tego badanego scroll nie uruchamia procesów poznawczych – scroll jest czynnikiem stresogennym.

Każde pociągnięcie palcem (szczególnie w Brainrocie) to dla niego wizualny "cios" (zapaść fali Alpha), który nie prowadzi do skupienia uwagi (brak górek Theta na czole), lecz podtrzymuje układ nerwowy i mięśniowy w stanie bardzo nieprzyjemnego napięcia (szum na Becie i drastycznie wysoka Theta ciemieniowa). Te wykresy to dobre zobrazowanie technologicznego "przebodźcowania".